# Project 05 · Insurance data quality and claim frequency

**Benson M. Merita · MeritaAnalytics** · Tools: Python (pandas, NumPy), Excel output

**Business question.** Which insurance records can be trusted, what is wrong with the rest, and what do the trusted records say about claim frequency by product?

**Data.** A policy extract (1,219 rows) and a claims extract (283 rows) from two systems. The true book has 1,200 policies and 265 claims; the extra rows and many errors were injected on purpose. Synthetic, generated with a fixed seed.

| Step | What it answers |
|---|---|
| 1 | Write the validation rules |
| 2 | Validate policies |
| 3 | Validate claims |
| 4 | Avoid the join trap that double-counts premium |
| 5 | Claim frequency, severity and loss ratio with intervals |
| 6 | Exception workbook for the operations team (Excel) |

## Setup

In [ ]:
from dataclasses import dataclass
from typing import Callable
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, DARK, AMBER = "#1F5FD6", "#0B2A4A", "#B45309"
plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)

DATA = Path("../datasets/05-insurance-quality/raw")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

## Step 1 · Write the validation rules

**What:** each rule is a small record with an ID, a description, a severity and a test. *Error* rows are quarantined; *Warning* rows are kept but sent for review.
**Why:** keeping rules as data means a manager can read the list, and a new rule can be added without touching the rest of the pipeline.

In [ ]:
@dataclass(frozen=True)
class Rule:
    rid: str
    table: str
    name: str
    severity: str          # "Error" or "Warning"
    test: Callable         # takes a table, returns True for each row that FAILS

def run_rules(df, rules):
    return pd.DataFrame({r.rid: r.test(df).fillna(False).astype(bool) for r in rules}, index=df.index)

def rule_report(flags, rules):
    return pd.DataFrame([{"rule": r.rid, "check": r.name, "severity": r.severity,
                          "rows_failing": int(flags[r.rid].sum())} for r in rules])

def failed_names(flags):
    return flags.apply(lambda row: ", ".join(row.index[row]), axis=1)

def has_error(flags, rules):
    return flags[[r.rid for r in rules if r.severity == "Error"]].any(axis=1)

## Step 2 · Validate policies

**What:** remove exact duplicates, parse dates, then run eight policy rules. "Premium not extreme" uses a cap per product: the upper quartile plus three times the interquartile range.
**Why:** claims can only be trusted if the policies they attach to are clean.

In [ ]:
policies = pd.read_csv(DATA / "policies.csv")
pol = policies.drop_duplicates().copy()
pol["start_dt"] = pd.to_datetime(pol["start_date"], format="%Y-%m-%d", errors="coerce")
pol["end_dt"] = pd.to_datetime(pol["end_date"], format="%Y-%m-%d", errors="coerce")
g = pol.groupby("product")["premium_ksh"]
q1, q3 = g.transform(lambda s: s.quantile(.25)), g.transform(lambda s: s.quantile(.75))
pol["cap"] = q3 + 3 * (q3 - q1)

pol_rules = [
    Rule("P01", "policies", "policy_id present", "Error", lambda d: d["policy_id"].isna()),
    Rule("P02", "policies", "policy_id is PL + 6 digits", "Error",
         lambda d: d["policy_id"].notna() & ~d["policy_id"].fillna("").str.fullmatch(r"PL\d{6}")),
    Rule("P03", "policies", "policy_id unique", "Error",
         lambda d: d["policy_id"].notna() & d["policy_id"].duplicated(keep=False)),
    Rule("P04", "policies", "dates valid", "Error", lambda d: d["start_dt"].isna() | d["end_dt"].isna()),
    Rule("P05", "policies", "end after start", "Error", lambda d: d["end_dt"] <= d["start_dt"]),
    Rule("P06", "policies", "premium above zero", "Error", lambda d: d["premium_ksh"] <= 0),
    Rule("P07", "policies", "customer_id present", "Warning", lambda d: d["customer_id"].isna()),
    Rule("P08", "policies", "premium not extreme", "Warning", lambda d: d["premium_ksh"] > d["cap"]),
]
pflags = run_rules(pol, pol_rules)
perr = has_error(pflags, pol_rules)
pol_clean = pol[~perr].copy()
n_pdup = len(policies) - len(pol)
print(f"policies: raw {len(policies)} = duplicates {n_pdup} + quarantined {int(perr.sum())} + clean {len(pol_clean)}")
rule_report(pflags, pol_rules)

## Step 3 · Validate claims

**What:** remove exact duplicates, look up each claim's policy (only from *clean* policies), then run eight claim rules.
**Why:** two problems an ID check alone would miss: **orphan** claims whose policy does not exist, and **re-keyed duplicates**, the same claim entered twice under a new claim ID (same policy, date and amount).

In [ ]:
claims = pd.read_csv(DATA / "claims.csv")
cl = claims.drop_duplicates().copy()
cl["claim_dt"] = pd.to_datetime(cl["claim_date"], format="%Y-%m-%d", errors="coerce")
cl["in_extract"] = cl["policy_id"].isin(pol["policy_id"].dropna())
lookup = pol_clean[["policy_id", "product", "county", "start_dt", "end_dt", "sum_insured_ksh"]]
cl = cl.merge(lookup, on="policy_id", how="left", validate="many_to_one")

key = ["policy_id", "claim_dt", "claim_amount_ksh"]
clm_rules = [
    Rule("K01", "claims", "policy_id present", "Error", lambda d: d["policy_id"].isna()),
    Rule("K02", "claims", "claim date valid", "Error", lambda d: d["claim_dt"].isna()),
    Rule("K03", "claims", "amount above zero", "Error", lambda d: d["claim_amount_ksh"] <= 0),
    Rule("K04", "claims", "policy exists (no orphan)", "Error", lambda d: d["policy_id"].notna() & ~d["in_extract"]),
    Rule("K05", "claims", "policy passed validation", "Error", lambda d: d["in_extract"] & d["product"].isna()),
    Rule("K06", "claims", "not a re-keyed duplicate", "Error",
         lambda d: d["policy_id"].notna() & d.duplicated(subset=key, keep="first")),
    Rule("K07", "claims", "date inside cover", "Warning",
         lambda d: d["product"].notna() & d["claim_dt"].notna() & ~d["claim_dt"].between(d["start_dt"], d["end_dt"])),
    Rule("K08", "claims", "amount within sum insured", "Warning",
         lambda d: d["product"].notna() & (d["claim_amount_ksh"] > d["sum_insured_ksh"])),
]
cflags = run_rules(cl, clm_rules)
cerr = has_error(cflags, clm_rules)
n_cdup = len(claims) - len(cl)
clm_ok = cl[~cerr & ~cflags["K07"] & ~cflags["K08"]].copy()      # accepted: no errors and no warnings
print(f"claims: raw {len(claims)} = duplicates {n_cdup} + quarantined {int(cerr.sum())} + clean {int((~cerr).sum())}")
print("claims accepted for analysis (no errors, no warnings):", len(clm_ok))
rule_report(cflags, clm_rules)

## Step 4 · Avoid the join trap

**What:** first, the *wrong* way: join claims onto policies and sum premium. A policy with two claims now appears twice, so its premium is counted twice. Then a merge with `validate="one_to_one"` shows pandas refusing the bad join. Finally the right way: **aggregate claims per policy first, then join one-to-one**.
**Why:** this mistake is common in reports and silently inflates premium and loss ratios.

In [ ]:
naive = pol_clean.merge(clm_ok[["policy_id", "claim_id"]], on="policy_id", how="left")
overstated = naive["premium_ksh"].sum() / pol_clean["premium_ksh"].sum() * 100 - 100
print(f"policies {len(pol_clean)} -> naive join rows {len(naive)} | premium overstated by {overstated:.2f}%")

try:
    pol_clean.merge(clm_ok, on="policy_id", how="left", validate="one_to_one")
except pd.errors.MergeError as e:
    print("validate='one_to_one' stopped the bad join:", e)

In [ ]:
per_policy = clm_ok.groupby("policy_id").agg(claims=("claim_id", "count"),
                                             claim_ksh=("claim_amount_ksh", "sum")).reset_index()
base = pol_clean.merge(per_policy, on="policy_id", how="left", validate="one_to_one")
base[["claims", "claim_ksh"]] = base[["claims", "claim_ksh"]].fillna(0)
base["exposure_years"] = ((base["end_dt"] - base["start_dt"]).dt.days + 1) / 365.25
print("row count kept:", len(base) == len(pol_clean),
      "| premium unchanged:", abs(base["premium_ksh"].sum() - pol_clean["premium_ksh"].sum()) < 0.01,
      "| every accepted claim carried through:", int(base["claims"].sum()) == len(clm_ok))

## Step 5 · Claim frequency with honest uncertainty

**What:** for each product: claims per 100 policy-years, an approximate 95% interval, average claim size (severity) and loss ratio (claims ÷ premium).
**Why:** small groups produce noisy rates. The interval shows which differences are real and which are luck.

In [ ]:
def segment_table(b, by):
    t = b.groupby(by).agg(policies=("policy_id", "count"), exposure=("exposure_years", "sum"),
                          claims=("claims", "sum"), premium=("premium_ksh", "sum"), claim_ksh=("claim_ksh", "sum"))
    t["freq_per_100py"] = t["claims"] / t["exposure"] * 100
    half = 1.96 * np.sqrt(t["claims"]) / t["exposure"] * 100
    t["ci_low"], t["ci_high"] = (t["freq_per_100py"] - half).clip(lower=0), t["freq_per_100py"] + half
    t["severity_ksh"] = t["claim_ksh"] / t["claims"].replace(0, np.nan)
    t["loss_ratio_pct"] = t["claim_ksh"] / t["premium"] * 100
    return t

by_prod = segment_table(base, "product").sort_values("freq_per_100py", ascending=False)
by_prod[["policies", "claims", "freq_per_100py", "ci_low", "ci_high", "severity_ksh", "loss_ratio_pct"]].round(1)

In [ ]:
fig, ax = plt.subplots()
t = by_prod
ax.bar(t.index, t["freq_per_100py"], color=[AMBER if v == t["freq_per_100py"].max() else BLUE for v in t["freq_per_100py"]],
       yerr=[t["freq_per_100py"] - t["ci_low"], t["ci_high"] - t["freq_per_100py"]], capsize=5, ecolor=DARK)
ax.set_title("Claim frequency by product, with approximate 95% interval")
ax.set_ylabel("Claims per 100 policy-years")
plt.tight_layout(); plt.show()

**County view.** The intervals for counties overlap heavily, so no county should be singled out:

In [ ]:
by_cty = segment_table(base, "county").sort_values("freq_per_100py", ascending=False)
by_cty[["claims", "freq_per_100py", "ci_low", "ci_high"]].round(1)

**How much of the true book survived?** The data documentation states the true book had 265 claims.

In [ ]:
TRUE_CLAIMS = 265     # from the dataset documentation
print(f"accepted {len(clm_ok)} of {TRUE_CLAIMS} true claims = {100 * len(clm_ok) / TRUE_CLAIMS:.1f}%; "
      "every rate above is therefore a lower bound")

## Step 6 · Exception workbook for the operations team

**What:** write an Excel file with three sheets: a **Scorecard** (rows failing each rule), **Policy exceptions** and **Claim exceptions** (each failing row with the rules it broke). Then read it back to confirm.
**Why:** the people who fix records work in Excel, not in Python.

In [ ]:
scorecard = pd.concat([rule_report(pflags, pol_rules), rule_report(cflags, clm_rules)], ignore_index=True)
pol_exc = pol[pflags.any(axis=1)].assign(failed_rules=failed_names(pflags)).drop(columns=["start_dt", "end_dt", "cap"])
clm_exc = (cl[cflags.any(axis=1)].assign(failed_rules=failed_names(cflags))
           .drop(columns=["claim_dt", "in_extract", "start_dt", "end_dt"]))
xl = OUT / "05_data_quality_exceptions.xlsx"
with pd.ExcelWriter(xl) as writer:
    scorecard.to_excel(writer, sheet_name="Scorecard", index=False)
    pol_exc.to_excel(writer, sheet_name="Policy exceptions", index=False)
    clm_exc.to_excel(writer, sheet_name="Claim exceptions", index=False)
back = pd.read_excel(xl, sheet_name=None)
print({sheet: frame.shape for sheet, frame in back.items()})
clm_exc[["claim_id", "policy_id", "claim_date", "claim_amount_ksh", "failed_rules"]].head()

## Key findings and recommendations

- **1,168 clean policies of 1,219** and **222 accepted claims of 283**. Re-keyed duplicate claims and orphan claims were caught.
- The naive join **overstates premium by about 3%**. Always aggregate first, then join one-to-one with a `validate` check.
- **Medical** has the highest claim frequency and **Motor Private** the highest loss ratio. Home and Personal Accident are too rare to rank with confidence.
- *Make policy and claim IDs mandatory at export, add a re-keyed duplicate check at claims intake, and route claims outside the cover period or above the sum insured to manual review.*

**Limitations.** Only about 84% of true claims survive validation, so every rate is a lower bound. Data is synthetic.